-----
# Data Cleaning
-----

### Notebook Summary

All extracted information will act as features in my dataset and will be used in embeddings, clustering and eventually, for building classification models.

In this notebook, I clean the data by removing punctuation, whitespace, and other irrelevant content. This cleaning step is needed to ensure that, before vectorisation, the data focuses solely on the actual email content, making results more accurate.

## Set Up
---

In [25]:
import numpy as np
import pandas as pd
import re
import string

## Functions
-----

In [26]:
def df_check (df):

    shape = df.shape
    nulls = df.isna().sum().sum()
    duplicated_rows = df.duplicated().sum()
    duplicated_cols = df.columns.duplicated().sum()


    print (f"""      
    Number of Rows: {shape[0]}     
    Number of Columns: {shape[1]}     
    Number of Nulls: {nulls}       
    Number of Duplicated Rows: {duplicated_rows}
    Number of Duplicated Cols: {duplicated_cols}
        """)

## Data Loading

In [27]:
emails_df  = pd.read_csv('../../data/processed_emails.csv', index_col=0)

In [28]:
emails_df.head()

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,NaN,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Randy,\n\n Can you send me a schedule of the s..."
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


In [29]:
df_check(emails_df)

      
    Number of Rows: 517401     
    Number of Columns: 4     
    Number of Nulls: 28333       
    Number of Duplicated Rows: 264855
    Number of Duplicated Cols: 0
        


### Nulls
----

In [30]:
emails_df.isna().sum()

from           0
to          9146
subject    19187
body           0
dtype: int64

#### 1. Null Recipients

In [31]:
emails_df[emails_df['to'].isna()]

,from,to,subject,body
1230,outlook-migration-team@enron.com,NaN,Lee Odonnel,loan servicing-jessica weeber 800-393-5626 jwe...
1231,outlook-migration-team@enron.com,NaN,Greg Thorse,exit mccollough off 410
3909,jeff.youngflesh@enron.com,NaN,Lexmark Document/Workflow mgmt intro,Lexmark Solution Services intro to GSS BD. Ga...
4395,john.arnold@enron.com,NaN,NaN,"To ""Outstanding"" Analysts and Associates:\nI ..."
4442,ann.schmidt@enron.com,NaN,Enron Mentions,Dabhol lenders to vote today on PPA PPPPA term...
...,...,...,...,...
516316,john.phillips@clarionenergy.com,NaN,Corrected Post-AGA NYMEX straddles,Sent wrong file previously-- We apologize
516731,andy.zipper@enron.com,NaN,"EDS Arcordia, set up meeting Kolodgie",Spoke with Kolodgie. Arcordia is run out of Lo...
516732,andy.zipper@enron.com,NaN,Red Meteor etc.,TASK ASSIGNMENT\n\n\nTask Priority:\t\t\nTask ...
516846,john.zufferli@enron.com,NaN,NaN,Conference call with UBS


-----
**Comment:**

After looking into the null recipients, cases where the to are missing either they are addressed to email groups or no recipient can be found. Making the decision to remove all cases where recipients are missing, for this project I think it important to work with a complete dataset for more accurate insights, models and results.

In [32]:
emails_df.dropna(subset=['to'], inplace=True)

In [33]:
emails_df.reset_index(drop=True, inplace=True)

#### 2. Null Subject

In [34]:
emails_df[emails_df['subject'].isna()]

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,NaN,Here is our forecast
3,phillip.allen@enron.com,randall.gay@enron.com,NaN,"Randy,\n\n Can you send me a schedule of the s..."
6,phillip.allen@enron.com,"david.l.johnson@enron.com, john.shafer@enron.com",NaN,Please cc the following distribution list with...
11,phillip.allen@enron.com,stagecoachmama@hotmail.com,NaN,"Lucy,\n\n Here are the rentrolls:\n\n\n\n Open..."
14,phillip.allen@enron.com,david.delainey@enron.com,NaN,"Dave, \n\n Here are the names of the west desk..."
...,...,...,...,...
507699,peggy.a.kostial@accenture.com,andy.zipper@enron.com,NaN,Thanks for the note below. Curious how you all...
507708,frank.hayden@enron.com,john.zufferli@enron.com,NaN,Sorry I missed call. My understanding is that...
508101,john.zufferli@enron.com,majordomo@majordomo.pjm,NaN,unsubscribe pjm-customer-info
508143,john.zufferli@enron.com,john.lavorato@enron.com,NaN,home number is (403) 685-4817


------
**Comment:** 

Making decision to also drop cases where email subject is missing from the dataset. Emails with missing subjects only make up 3% of the dataset and for me its more imporant to have a complete dataset for this project and 3% data loss is not a big sacrifice to me. 

In [35]:
emails_df.dropna(subset=['subject'], inplace=True)

In [36]:
emails_df.reset_index(drop=True, inplace=True)

#### ReCheck of Nulls

In [37]:
df_check(emails_df)

      
    Number of Rows: 489243     
    Number of Columns: 4     
    Number of Nulls: 0       
    Number of Duplicated Rows: 251032
    Number of Duplicated Cols: 0
        


### Duplicated Emails
-----

In [38]:
emails_df[emails_df.duplicated()]

,from,to,subject,body
414,phillip.allen@enron.com,keith.holst@enron.com,Consolidated positions: Issues & To Do list,---------------------- Forwarded by Phillip K ...
415,phillip.allen@enron.com,keith.holst@enron.com,Consolidated positions: Issues & To Do list,---------------------- Forwarded by Phillip K ...
416,phillip.allen@enron.com,paula.harris@enron.com,Re: 2001 Margin Plan,"Paula,\n\n 35 million is fine\n\nPhillip"
417,phillip.allen@enron.com,ina.rangel@enron.com,"Var, Reporting and Resources Meeting",---------------------- Forwarded by Phillip K ...
418,phillip.allen@enron.com,pallen70@hotmail.com,Westgate,---------------------- Forwarded by Phillip K ...
...,...,...,...,...
489144,john.zufferli@enron.com,cooper.richey@enron.com,FW: Ontario overview,"-----Original Message-----\nFrom: \tTripp, Gar..."
489164,john.zufferli@enron.com,daniel.diamond@enron.com,RE: Canadian Online Counterparties,Here is the list updated with responsibility n...
489181,john.zufferli@enron.com,chris.dorland@enron.com,FW: 1996 Porsche 993 C4S,"i lifted it last night for 82,000\n -----Origi..."
489199,john.zufferli@enron.com,chris.dorland@enron.com,1998 Silver C4S,Check out www.segalmotorcar.com


----
**Comment:**

Making the decision to drop all duplicated emails.

In [39]:
emails_df.drop_duplicates(inplace=True)

In [40]:
emails_df.reset_index(drop= True, inplace=True)

In [41]:
emails_df

,from,to,subject,body
0,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
1,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
2,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.
3,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,"Greg,\n\n How about either next Tuesday or Thu..."
4,phillip.allen@enron.com,joyce.teixeira@enron.com,Re: PRC review - phone calls,any morning between 10 and 11:30
...,...,...,...,...
238206,john.zufferli@enron.com,kori.loibl@enron.com,Trade with John Lavorato,This is a trade with OIL-SPEC-HEDGE-NG (John L...
238207,john.zufferli@enron.com,john.lavorato@enron.com,Gas Hedges,Some of my position is with the Alberta Term b...
238208,john.zufferli@enron.com,dawn.doucet@enron.com,RE: CONFIDENTIAL,2\n\n -----Original Message-----\nFrom: \tDouc...
238209,john.zufferli@enron.com,jeanie.slone@enron.com,Calgary Analyst/Associate,Analyst\t\t\t\t\tRank\n\nStephane Brodeur\t\t\...


#### ReCheck of Duplicates

In [42]:
df_check(emails_df)

      
    Number of Rows: 238211     
    Number of Columns: 4     
    Number of Nulls: 0       
    Number of Duplicated Rows: 0
    Number of Duplicated Cols: 0
        


## Sampling to get subset of emails
----

In [43]:
sampled_emails = emails_df.sample(n=30_000, random_state=11)

## Cleaning Email Body and Subject
------

Cleaning email body and subject by removing:

- new line and tab characters
- email headers (like to, from, subject, sent)
- punctuation

In [44]:
from bs4 import BeautifulSoup

def clean_email_with_soup(email):
    """
    Description: 
        Clean email data to address noise such as HTML tags, email addresses, phone numbers, files, etc.

    Input:
        Raw email data for a single email.

    Output:
        Cleaned email data for a single email.
    """
    # Use BeautifulSoup to remove all HTML tags
    if '>' in email or  '<' in email:
        soup = BeautifulSoup(email, 'html.parser')
        email = soup.get_text()

    # Remove email headers (forwarded by/original message, from, to, subject, sent)
    email = re.sub(r'[-\s]?(Forwarded by|Original Message)(\s*.*?)?[-\s]+', '', email)
    email = re.sub(r'From:.*?(\n|$)', '', email)
    email = re.sub(r'To:.*?(\n|$)', '', email)
    email = re.sub(r'Subject:.*?(\n|$)', '', email)
    email = re.sub(r'Sent:.*?(\n|$)', '', email)

    # Remove email addresses and phone numbers 
    email = re.sub(r'\S+@\S+', '', email)
    email = re.sub(r'(\(?\d{3}\)?[-.\s]?)\d{3}[-.\s]?\d{4}', '', email)

    # Remove files and filepaths
    email = re.sub(r'\b[\w.-]+(?:\.(?:docx?|xlsx?|pdf|txt|html|zip|rar|png|jpe?g|gif))\b', '', email)
    email = re.sub(r'[\w]*FilePath\S*', '', email)   
    
    # Remove web addresses
    email = re.sub(r'http[s]?://\S+|www\.\S+', '', email)

    # Remove promotional and ad language
    promo_patterns = [
        r'(?i)(unsubscribe|opt-out|sent from|advertisement|sponsored by).*',
        r'(?i)(free\s+(trial|issue|gift|offer|camera|subscription)|limited\s+time\s+offer|act\s+now|subscribe\s+now|you\s+have\s+been\s+selected).*',
        r'(?i)(save\s+up\s+to|absolutely\s+free|most\s+authoritative|exclusive\s+offer).*',
        r'(?i)(click\s+here|call\s+now|follow\s+the\s+link|order\s+now|subscribe\s+today).*',
        r'(?i)(this\s+email\s+was\s+sent\s+to|view\s+in\s+browser|privacy\s+policy).*',
        r'(?i)\b(image|img)\b.*'
    ]
    for pattern in promo_patterns:
        email = re.sub(pattern, '', email)


    # Remove non-alphanumeric symbols except for spaces
    email = re.sub(r'[^\w\s]|_', '', email)
 
    # Remove non-ASCII characters
    email = re.sub(r'[^\x00-\x7F]+', '', email)

    # Remove extra whitespace
    email = re.sub(r'\s+', ' ', email).strip()

    return email

In [45]:
sampled_emails['cleaned_body'] = sampled_emails['body'].apply(clean_email_with_soup)

/var/folders/m1/502kn7jx4nd1sm29zp721mh40000gn/T/ipykernel_61262/1144272986.py:16: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(email, 'html.parser')


----
**Comment:**

`MarkupResemblesLocatorWarning: The input looks more like a URL than markup. You may want to use an HTTP client like requests to get the document behind the URL, and feed that document to Beautiful Soup`

Warning could be due to BeautifulSoup thinking input may be a filename or URL rather than HTML, and so added in a if statement to say if input contains tags (<>) then to use BeautifulSoup to remove them if not pass over BeautifulSoup call.


`MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(email, 'html.parser')`

This could be due to the emails where there are email attachemnts, I have a regular expression to remove file paths and files so am choosing to ignore this error.

In [46]:
sampled_emails['cleaned_subject'] = sampled_emails['subject'].apply(clean_email_with_soup)

## Export cleaned data
----

In [47]:
# exporting df to csv, ready for next stage
sampled_emails.to_csv('../../data/cleaned_emails.csv')

## Summary
----

With the email data now cleaned, I can progress to vectorising the text data using word embeddings to better capture the context behind emails. This is important as these insights will guide me when clustering.